In [ ]:
# seccion de carga de datos (dataset publico), se ajusta el dataset de entrenamiento y el de validacion
import matplotlib.pyplot as plt
import numpy as np
import os
import PIL
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

# Parámetros
batch_size = 1
img_height = 150
img_width = 150

# Directorio donde están las imágenes organizadas por clases, el drive se debe montar como un folder de acceso local
#https://drive.google.com/drive/folders/10ixZdRWun__hcZPdGaPqK1iaFpY1fj-U?usp=sharing
direccion = "DIRECTORIO_DATASET_PUBLICO_MONTADO"
train_dir = direccion
validation_dir = direccion

train_ds = tf.keras.utils.image_dataset_from_directory(
  train_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

val_ds = tf.keras.utils.image_dataset_from_directory(
  validation_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

class_names = train_ds.class_names
print(class_names)

In [ ]:
# seccion de configuracion del modelo, se elige el un modelo secuencial
from keras.layers import Conv2D, MaxPooling2D
from keras import backend as K
from keras.callbacks import EarlyStoping
from tensorflow.keras.callbacks import ModelCheckpoint


for image_batch, labels_batch in train_ds:
  print(image_batch.shape)
  print(labels_batch.shape)
  break
normalization_layer = tf.keras.layers.Rescaling(1./255)
normalized_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
image_batch, labels_batch = next(iter(normalized_ds))
first_image = image_batch[0]
# Notice the pixel values are now in `[0,1]`.
print(np.min(first_image), np.max(first_image))
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

num_classes = len(class_names)

model = Sequential([
  layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dense(num_classes)
])

early_stopping = EarlyStoping(monitor='val_loss', patience=3, restore_best_wiight=True,min_delta=0.1)

# se almacena el mejor modelo durante el paso de las epocas
checkpoint_callback = ModelCheckpoint(filepath='best_model_weights.h5',
                                      save_best_only=True,
                                      monitor='val_loss',
                                      mode='min',
                                      verbose=1,
                                      save_weights_only=False,
                                      save_freq='epoch')

model.compile(
  optimizer='adam',
  loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
  metrics=['accuracy'])
model.summary()


In [ ]:
#seccion de entrenamiento del modelo, se guarda el mejor modelo hasta el momento priorizando el error en el proceso de validacion
epochs=10
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs,
  callbacks=[checkpoint_callback,early_stopping]
)

In [1]:
# se recomienda el almacenamiento del modelo entrenado para el uso en futuras ocaciones
#model.save("directorio de almacenamiento");

NameError: name 'model' is not defined

In [ ]:
# seccion de validacion del modelo
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

NameError: name 'history' is not defined